# Approximate QFT on 6 Qubits

In this Jupyter notebook, we implement AQFT Circuit 1 from Park-Ahn, with $n $ qubits, error $\varepsilon $. 

In [2]:
from qrisp import QuantumSession, QuantumVariable, QuantumCircuit, prepare, h, z, s, gidney_adder, QFT

import matplotlib.pyplot as plt

import numpy as np

# The input Qubit

We initialize the circuit with a quantum register containing (n) qubits and a prepared ancilla register containing (b+1) qubits, where

$$
b = \left\lceil \log_2\left(\frac{n}{\epsilon}\right) \right\rceil.
$$

Therefore, the number of ancilla qubits required is

$$
b+1 = \left\lceil \log_2\left(\frac{n}{\epsilon}\right) \right\rceil + 1.
$$



In [3]:
n = 3 #number of qubits                         # n = number of qubits we are applying the AQFT to
epsilon = 0.2                                   # epsilon = spectral norm error tolerance for the approximation
b = int(np.ceil(np.log2(n/epsilon)))            # b + 1 = number of ancilla qubits needed


In [4]:
qs = QuantumSession()

quantumRegister = QuantumVariable(n, name='q', qs = qs)          # quantum register of n qubits
ancillaRegister = QuantumVariable(int(b+1), name='a', qs = qs)   # ancilla register of b+1 qubits

In [12]:
# Input state preparation
amplitudes = np.random.randint(0, high=10, size=2**(n), dtype=int)  # generate random amplitudes for the input state
psi = prepare(quantumRegister, amplitudes)  

## Fan Out

The following method applied a fan-out gate as defined in Park-Ahn, which is a sequence of CX gates controlled by a single qubit.

In [6]:
def fan_out(circuit, qubit_index, ancilla_indices):
    """Fan-out operation using CNOT gates.
    
        Input: circuit -> QuantumCircuit object
               qubit_index -> index of the qubit to fan out from
               ancilla_indices -> list of indices of the ancilla qubits to fan out to

        Output: None
    
    """
    circuit.cx(qubit_index, ancilla_indices)

    return None

# Assumed access to the prepared state $|phi\rangle$
We assume that we have access to the correct state for ancilla register.

In [7]:
# This is done by preparing the ancilla register in the state |1>^(b+1) using the prepare function from qrisp.
allOnesAmplitudes = np.array([0]*(2**(b+1) - 1) + [1], dtype=float)
allOnes = prepare(ancillaRegister, allOnesAmplitudes)  # prepare the ancilla register in the state |1>^(b+1)

# We apply inverse Quantum Fourier Transform to the all ones state to prepare the ancilla correctly for use with Gidney adder
QFT(ancillaRegister, inv=True, qiskit_endian=False)  # apply the inverse QFT to the ancilla register

<QuantumVariable 'a'>

## AQFT Circuit : # Circuit 1 (T count reduction) 

In [8]:
b = n - 1

# -------------------------
# Circuit Block 1
# -------------------------

h(quantumRegister[0])
z(quantumRegister[0])

for i in range(1, n):
    s(quantumRegister[i])

gidney_adder(
    quantumRegister[0:b - 1],
    ancillaRegister
)

fan_out(
    qs,
    quantumRegister[0],
    quantumRegister[1:b]
)

gidney_adder(
    quantumRegister[0:b - 1],
    ancillaRegister
)

fan_out(
    qs,
    quantumRegister[0],
    quantumRegister[1:b]
)


# -------------------------
# Repeated circuit blocks
# -------------------------

for control_index in range(1, b):
    h(quantumRegister[control_index])
    s(quantumRegister[control_index])

    targets = quantumRegister[control_index + 1:n]

    # Do not call fan_out when there are no target qubits.
    if len(targets) > 0:
        fan_out(
            qs,
            quantumRegister[control_index],
            targets
        )

    gidney_adder(
        quantumRegister[control_index:n],
        ancillaRegister
    )

    if len(targets) > 0:
        fan_out(
            qs,
            quantumRegister[control_index],
            targets
        )


# -------------------------
# Last qubit
# -------------------------

h(quantumRegister[b])
s(quantumRegister[b])


# -------------------------
# Final circuit block
# -------------------------

for qubit in quantumRegister:
    s(qubit)

gidney_adder(
    quantumRegister[-1:-b:-1],
    ancillaRegister
)



In [9]:

# Convert to Qiskit and draw
qc = qs.to_qiskit()

qc.draw(
    output="mpl",
    style={"backgroundcolor": "#EEEEEE"},
    fold=100,
    idle_wires=False
)

# testing 